# 示例策略

演示 StrategyDef 框架的 notebook 用法

本 notebook 展示了一个基于均线交叉的简单交易策略：

1. __STRATEGY_META__ 元信息声明
2. MakeStrategy 子类 —— 重写 genSignal()
3. 直接构建策略对象

策略逻辑：当短期均线上穿长期均线时买入（信号=1），下穿时卖出（信号=-1）。

In [ ]:
import pandas as pd

from QuantStudio.Factor import FactorOperator as fo
from QuantStudio.Factor.BasicOperator import rename
from QuantStudio.Factor.HDF5DB import HDF5DB
from QuantStudio.BackTest.Strategy.Strategy import MakeStrategy, MakeAccount


# ============================================================
# __STRATEGY_META__ — 模块元信息
# ============================================================
__STRATEGY_META__ = {
    # ---- 必填 ----
    "TargetTable": "strategy_signals_example",
    "IDType": "A股",
    "Description": "均线交叉策略示例：短期均线上穿长期均线买入，下穿卖出",

    # ---- 算子默认配置 ----
    "OperatorConfig": {
        "SignalType": "目标权重",
        "InitCash": 1e6,
        "ShortAllowed": False,
    },

    # ---- 依赖声明 ----
    "FactorDeps": {
        "stock_cn_day_bar_nafilled": ["close"],
    },
    "DBDeps": {"JYDB": "聚源数据库"},

    # ---- 可调参数 ----
    "ModelArgs": {
        "short_window": "短期均线窗口",
        "long_window": "长期均线窗口",
    },

    "Author": "示例作者",
    "Tags": ["示例", "均线", "趋势跟踪"],
    "MaxLookBack": 120,
}


# ============================================================
# MakeStrategy 子类
# ============================================================

class MACrossStrategy(MakeStrategy):
    """均线交叉策略

    信号逻辑：
      - ma_short > ma_long → 买入 (signal=1)
      - ma_short < ma_long → 卖出 (signal=-1)
      - 否则 → 不变 (signal=0)
    """

    def genSignal(self, f, idt, x, last_price, cash, position_num, args):
        """生成交易信号

        Args:
            f: 当前策略因子
            idt: 当前时点
            x: 输入的因子数据列表，x[0]=short_ma, x[1]=long_ma
            last_price: 最新价格
            cash: 现金
            position_num: 当前持仓数量
            args: 策略参数

        Returns:
            pd.Series: 信号值，索引为证券ID
        """
        short_ma = x[0]
        long_ma = x[1]

        if short_ma is None or long_ma is None:
            return None

        signal = pd.Series(0, index=short_ma.index)
        signal[short_ma > long_ma] = 1
        signal[short_ma < long_ma] = -1

        return signal

In [ ]:
# ============================================================
# 策略参数
# ============================================================
short_window = 5
long_window = 20
signal_type = "目标权重"
init_cash = 1e6
short_allowed = False

# ============================================================
# 构建策略因子
# ============================================================

# 数据库连接
HDB = HDF5DB(args={"MainDir": r"D:\Data\HDF5DB"}).connect()

# 获取收盘价
FT = HDB.getTable("stock_cn_day_bar_nafilled")
close = FT.getFactor("close")

# 计算短期和长期均线
ma_short = fo.RollingMean(window=short_window, min_periods=1)(close)
ma_long = fo.RollingMean(window=long_window, min_periods=1)(close)

# 构造策略实例（x_lookback 指定每个依赖因子的回溯期数）
makeStrategy = MACrossStrategy(
    signal_type=signal_type,
    init_cash=init_cash,
    short_allowed=short_allowed,
    x_lookback=[short_window - 1, long_window - 1],
    x_section_ids=[None, None],
)

# 调用策略算子，传入均线因子和价格因子
Account = makeStrategy(ma_short, ma_long, last_price=close)

---
## 交互式调试

以下 cell 无 `strategy-def` tag，框架加载时不会执行。
用于本地数据探索和可视化。

In [ ]:
# 回测示例
from QuantStudio.Factor.JYDB import JYDB
from QuantStudio.Core.Node import DTLocalContext, DTInitData
from QuantStudio.Factor.Factor import FactorContext
from QuantStudio.Core.CalcEngine import Engine
from QuantStudio.Core import setDefaultLogLevel
import logging; setDefaultLogLevel(logging.INFO)

SDB = JYDB().connect()
DTs = SDB.getTradeDay(start_date=pd.to_datetime("2024-01-01"), end_date=pd.to_datetime("2024-12-31"))
IDs = SDB.getStockID(is_current=True)

with FactorContext(PIDList=["0"], DTRuler=DTs, SectionIDs=IDs) as ctx:
    with Engine() as eng:
        Output = eng.run([Account], ctx, fwd_data_list=[DTLocalContext(DTs=DTs)])
print(Output[0]["统计数据"])